# Podstawy sztucznej inteligencji — Laboratorium 3
## Wiedza, wnioskowanie i grafy wiedzy

Na wykładzie 4 zobaczyliśmy, jak zapisać wiedzę w maszynie i jak wyciągać z niej wnioski. Dziś **nie piszemy algorytmów od zera**. Wszystkie funkcje są gotowe i opisane komentarzami. Waszym zadaniem jest:

- **czytać** gotowy kod i rozumieć, co robi,
- **przewidywać** wynik, zanim uruchomicie komórkę,
- **zmieniać wiedzę**: dopisywać fakty, reguły, trójki i liczby,
- **odpowiadać** krótko na pytania w komórkach tekstowych.

| Część | Temat | Czas |
|---|---|---|
| 1 | Baza wiedzy i wnioskowanie w przód | 20 min |
| 2 | Wnioskowanie wstecz | 25 min |
| 3 | Wielu studentów, świat zamknięty i otwarty | 10 min |
| 4 | Ontologia jako trójki i reasoner | 30 min |
| 5 | Wikidata i SPARQL | 20 min |

**Jak pracujemy**
- Najpierw zapisz kopię: *Plik → Zapisz kopię na Dysku*.
- Uruchamiaj komórki po kolei (Shift+Enter). Każda część zaczyna się od komórki **„Gotowe narzędzia”**: uruchom ją i przeczytaj komentarze.
- Miejsca do zmiany są oznaczone `# TODO`. Zwykle to jedna linia: fakt, reguła albo liczba.
- Po zadaniu często jest komórka z testem. Gdy wszystko działa, zobaczysz ✅.
- Zadania z ★ są dla chętnych.
- Przykład przewodni jest ten sam co na wykładzie: uczelnia, Ania, PSI, Kowalski.

In [1]:
# Uruchom tę komórkę na początku: importy bibliotek
import random              # losowa kolejność reguł (zadanie 2.3)
import pandas as pd        # wyniki w postaci tabel

def ok(nazwa):
    # Wypisuje komunikat, gdy test przeszedł.
    print(f"✅ {nazwa}: wszystko działa")

print("Gotowe")

Gotowe


---
# Część 1. Baza wiedzy i wnioskowanie w przód

### Jak zapisujemy wiedzę?

- **Fakt** to napis, np. `"OddałaZadania"`. Wszystkie fakty trzymamy w zbiorze (`set`).
- **Reguła** to trójka `(nazwa, lista przesłanek, wniosek)`. Czytamy: *jeśli wszystkie przesłanki są prawdą, to wniosek jest prawdą*.

Baza wiedzy o Ani z wykładu (slajd 22):

| Reguła | Przesłanki → wniosek |
|---|---|
| R1 | OddałaZadania ∧ ZdałaKolokwium → ZaliczonyLab |
| R2 | ZaliczonyLab → Dopuszczona |
| R3 | ZaliczonyLab → MożeOddaćProjekt |
| R4 | Dopuszczona ∧ ZdałaEgzamin → ZaliczonyPrzedmiot |

In [3]:
# Fakty o Ani: to, co wiemy na pewno
FAKTY = {"OddałaZadania", "ZdałaKolokwium"}

# Reguły: (nazwa, [przesłanka 1, przesłanka 2, ...], wniosek)
REGULY = [
    ("R1", ["OddałaZadania", "ZdałaKolokwium"], "ZaliczonyLab"),
    ("R2", ["ZaliczonyLab"], "Dopuszczona"),
    ("R3", ["ZaliczonyLab"], "MożeOddaćProjekt"),
    ("R4", ["Dopuszczona", "ZdałaEgzamin"], "ZaliczonyPrzedmiot"),
]

In [4]:
# ===== Gotowe narzędzia: wnioskowanie w przód =====

def wnioskuj_w_przod(fakty, reguly, pokaz=True):
    '''
    Wnioskowanie w przód (slajd 23).

    Parametry:
        fakty  - zbiór faktów, od których zaczynamy
        reguly - lista reguł (nazwa, przesłanki, wniosek)
        pokaz  - True: wypisuj kolejne kroki, False: licz po cichu

    Zwraca:
        zbiór wszystkich faktów: tych na starcie i wywnioskowanych.
    '''
    fakty = set(fakty)    # robimy kopię, żeby nie zmienić zbioru podanego z zewnątrz
    runda = 0

    while True:           # powtarzamy, aż nic nowego się nie pojawi
        runda += 1
        nowe = set()      # nowe fakty znalezione w tej rundzie

        # Przechodzimy przez wszystkie reguły po kolei
        for nazwa, przeslanki, wniosek in reguly:
            # Reguła "odpala", gdy:
            #  1) wszystkie jej przesłanki są już faktami,
            #  2) jej wniosku jeszcze nie znamy (inaczej nic nowego nie wnosi).
            if all(p in fakty for p in przeslanki) and wniosek not in fakty:
                nowe.add(wniosek)
                if pokaz:
                    print(f"Runda {runda}: {nazwa} odpala → {wniosek}")

        # Punkt stały: w całej rundzie nie pojawił się żaden nowy fakt -> koniec
        if not nowe:
            if pokaz:
                print(f"Runda {runda}: nic nowego, STOP")
            return fakty

        # Dopisujemy nowe fakty do bazy dopiero po rundzie
        # (dlatego w rundzie 2 odpalają naraz R2 i R3, jak na slajdzie 25)
        fakty |= nowe

print("Narzędzia z części 1 gotowe")

Narzędzia z części 1 gotowe


### Zadanie 1.1. Przewidź, potem uruchom

Zanim uruchomisz komórkę, zapisz w komórce tekstowej poniżej:
1. Ile rund wykona algorytm?

2. Które fakty zostaną wywnioskowane?

3. Dlaczego R4 nie odpali?

Potem uruchom i porównaj ze slajdami 24–26.

In [8]:
wynik = wnioskuj_w_przod(FAKTY, REGULY)
print()
print("Wszystkie fakty:", sorted(wynik))
print("Nowe fakty:     ", sorted(wynik - FAKTY))

Runda 1: R1 odpala → ZaliczonyLab
Runda 2: R2 odpala → Dopuszczona
Runda 2: R3 odpala → MożeOddaćProjekt
Runda 3: nic nowego, STOP

Wszystkie fakty: ['Dopuszczona', 'MożeOddaćProjekt', 'OddałaZadania', 'ZaliczonyLab', 'ZdałaKolokwium']
Nowe fakty:      ['Dopuszczona', 'MożeOddaćProjekt', 'ZaliczonyLab']


**Moje przewidywania (1.1):**
1. 3 razy
2. ZaliczonyLab, Dopuszczona, MożeOddaćProjekt
3. Brakuje wymaganych przeslanek

### Zadanie 1.2. TELL: dodaj fakt

Agent oparty na wiedzy ma dwie operacje: **TELL** (dodaj wiedzę) i **ASK** (zapytaj). Wiedza jest oddzielona od programu: zmieniamy dane, a nie kod funkcji.

Ania zdała egzamin. Dopisz ten fakt do zbioru `FAKTY_2` i uruchom wnioskowanie jeszcze raz. W której rundzie odpala R4?

In [11]:
FAKTY_2 = FAKTY | set([
   "ZdałaEgzamin",
])
wnioskuj_w_przod(FAKTY_2, REGULY);

Runda 1: R1 odpala → ZaliczonyLab
Runda 2: R2 odpala → Dopuszczona
Runda 2: R3 odpala → MożeOddaćProjekt
Runda 3: R4 odpala → ZaliczonyPrzedmiot
Runda 4: nic nowego, STOP


In [12]:
# Test 1.2
assert "ZaliczonyPrzedmiot" in wnioskuj_w_przod(FAKTY_2, REGULY, pokaz=False), "Ania powinna zaliczyć przedmiot"
ok("Zadanie 1.2")

✅ Zadanie 1.2: wszystko działa


### Zadanie 1.3. Dopisz własne reguły

Dopisz **co najmniej 3 reguły** tak, żeby z bazy wynikał fakt `"Stypendium"`. Reguły mają tworzyć **łańcuch**: przesłanka jednej reguły to wniosek innej. Przykładowy pomysł: zaliczony przedmiot i średnia powyżej 4,5 → wysoka średnia roczna → stypendium. Jeśli Twoja reguła potrzebuje nowego faktu (np. o średniej), dopisz go do `MOJE_FAKTY`.

Sprawdź, w której rundzie odpala każda Twoja reguła.

In [14]:
MOJE_FAKTY = FAKTY_2 | set([
    "ŚredniaPowyżej4,5",
    # TODO: np. "ŚredniaPowyżej4,5",
])

MOJE_REGULY = [
("R5", ["ZaliczonyPrzedmiot", "ŚredniaPowyżej4,5"], "WysokaŚredniaRoczna"),
    ("R6", ["WysokaŚredniaRoczna"], "KandydatNaStypendium"),
    ("R7", ["KandydatNaStypendium"], "Stypendium")
]

wnioskuj_w_przod(MOJE_FAKTY, REGULY + MOJE_REGULY);

Runda 1: R1 odpala → ZaliczonyLab
Runda 2: R2 odpala → Dopuszczona
Runda 2: R3 odpala → MożeOddaćProjekt
Runda 3: R4 odpala → ZaliczonyPrzedmiot
Runda 4: R5 odpala → WysokaŚredniaRoczna
Runda 5: R6 odpala → KandydatNaStypendium
Runda 6: R7 odpala → Stypendium
Runda 7: nic nowego, STOP


In [15]:
# Test 1.3
assert len(MOJE_REGULY) >= 3, "Dopisz co najmniej 3 reguły"
assert "Stypendium" in wnioskuj_w_przod(MOJE_FAKTY, REGULY + MOJE_REGULY, pokaz=False), "Z bazy nie wynika Stypendium"
wnioski = {w for _, _, w in MOJE_REGULY}
assert any(p in wnioski for _, prz, _ in MOJE_REGULY for p in prz), "Reguły mają tworzyć łańcuch"
ok("Zadanie 1.3")

✅ Zadanie 1.3: wszystko działa


---
# Część 2. Wnioskowanie wstecz

Teraz zaczynamy od **pytania** (slajdy 27–30). Przeczytaj komentarze w funkcji. Zwróć uwagę na trzy rzeczy:
- funkcja wywołuje samą siebie (rekurencja), tak jak DFS z wykładu 2,
- gdy jedna reguła zawiedzie, próbuje następnej (nawrót),
- pamięta, które cele już są „w drodze”, żeby nie wpaść w pętlę.

In [16]:
# ===== Gotowe narzędzia: wnioskowanie wstecz =====

def wnioskuj_wstecz(cel, fakty, reguly, pokaz=True, _glebokosc=0, _w_drodze=frozenset()):
    '''
    Wnioskowanie wstecz (slajd 27).

    Parametry:
        cel    - fakt, o który pytamy, np. "Dopuszczona"
        fakty  - zbiór znanych faktów
        reguly - lista reguł (nazwa, przesłanki, wniosek)
        pokaz  - True: wypisz drzewo celów z wcięciami

    Zwraca:
        True, jeśli cel da się udowodnić, w przeciwnym razie False.

    Parametry z podkreśleniem (_glebokosc, _w_drodze) ustawia sama funkcja,
    gdy wywołuje się rekurencyjnie. Nie trzeba ich podawać.
    '''
    wciecie = "  " * _glebokosc          # im głębiej w drzewie, tym większe wcięcie
    def log(tekst):
        if pokaz:
            print(wciecie + tekst)

    # Krok 1: cel jest faktem -> udowodniony od razu
    if cel in fakty:
        log(f"{cel}: fakt ✓")
        return True

    # Ochrona przed cyklem: jeśli ten cel już próbujemy udowodnić wyżej w drzewie,
    # to kręcilibyśmy się w kółko (A potrzebuje B, B potrzebuje A, ...)
    if cel in _w_drodze:
        log(f"{cel}: cykl, przerywam ✗")
        return False

    # Krok 2: szukamy reguł, które mają ten cel jako wniosek
    kandydaci = [r for r in reguly if r[2] == cel]
    if not kandydaci:
        log(f"{cel}: brak faktu i reguły ✗")
        return False

    # Krok 3: próbujemy reguł po kolei. Gdy jedna zawiedzie, bierzemy następną (nawrót)
    for nazwa, przeslanki, _ in kandydaci:
        log(f"{cel}? próbuję {nazwa}: potrzebuję {' i '.join(przeslanki)}")
        # Każda przesłanka staje się nowym celem (rekurencja, o jeden poziom głębiej).
        # all() zatrzymuje się na pierwszej przesłance, której nie da się udowodnić.
        if all(wnioskuj_wstecz(p, fakty, reguly, pokaz, _glebokosc + 1, _w_drodze | {cel})
               for p in przeslanki):
            log(f"{cel}: TAK ({nazwa})")
            return True

    # Krok 4: żadna reguła się nie udała
    log(f"{cel}: NIE")
    return False


def wyjasnij(cel, fakty, reguly, _w_drodze=frozenset()):
    '''
    Moduł wyjaśnień (slajd 7): odpowiada na pytanie "dlaczego?".

    Działa jak wnioskuj_wstecz, ale zamiast True/False zwraca
    listę zdań z dowodem albo None, gdy celu nie da się udowodnić.
    '''
    if cel in fakty:
        return [f"{cel}: fakt"]
    if cel in _w_drodze:
        return None
    for nazwa, przeslanki, wniosek in reguly:
        if wniosek != cel:
            continue
        dowod = [f"{cel}, bo {nazwa}: {' i '.join(przeslanki)}"]
        udalo_sie = True
        for p in przeslanki:
            pod = wyjasnij(p, fakty, reguly, _w_drodze | {cel})   # dowód przesłanki
            if pod is None:
                udalo_sie = False
                break
            dowod += pod
        if udalo_sie:
            return dowod
    return None


def porownaj_algorytmy(lancuchy=100, dlugosc=10):
    '''
    Buduje dużą bazę wiedzy i porównuje pracę obu algorytmów (zadanie 2.3).

    Baza: `lancuchy` niezależnych łańcuchów reguł, każdy o długości `dlugosc`:
        L0_0 → L0_1 → ... → L0_10,   L1_0 → L1_1 → ...,   itd.
    Pytamy tylko o koniec jednego łańcucha.

    Zwraca słownik z liczbą prób zastosowania reguły w każdym algorytmie.
    '''
    fakty = {f"L{i}_0" for i in range(lancuchy)}                  # początki łańcuchów
    reguly = [(f"R{i}_{k}", [f"L{i}_{k}"], f"L{i}_{k+1}")
              for i in range(lancuchy) for k in range(dlugosc)]
    random.shuffle(reguly)                                        # reguły w losowej kolejności
    cel = f"L0_{dlugosc}"

    # Liczymy próby: w przód każda runda sprawdza wszystkie reguły.
    proby_przod = 0
    f = set(fakty)
    while True:
        proby_przod += len(reguly)
        nowe = {w for _, p, w in reguly if all(x in f for x in p)} - f
        if not nowe:
            break
        f |= nowe

    # Wstecz próbujemy tylko reguł na drodze do celu: jedna reguła na każdy krok łańcucha.
    proby_wstecz = dlugosc if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False) else None

    return {"reguł w bazie": len(reguly),
            "faktów wywnioskowanych w przód": len(f - fakty),
            "próby w przód": proby_przod,
            "próby wstecz": proby_wstecz}

print("Narzędzia z części 2 gotowe")

Narzędzia z części 2 gotowe


### Zadanie 2.1. Zadaj pytania

Uruchom komórkę i porównaj wydruk ze slajdami 28–30. Odpowiedz:
1. Których reguł algorytm w ogóle nie sprawdzał, gdy pytaliśmy o `Dopuszczona`?
2. Dlaczego odpowiedź na `ZaliczonyPrzedmiot` to NIE? Czy to znaczy, że Ania oblała egzamin?
3. Dopisz trzecie pytanie: czy Ania może oddać projekt?

In [17]:
print(wnioskuj_wstecz("Dopuszczona", FAKTY, REGULY))
print()
print(wnioskuj_wstecz("ZaliczonyPrzedmiot", FAKTY, REGULY))
print()
# TODO: trzecie pytanie

Dopuszczona? próbuję R2: potrzebuję ZaliczonyLab
  ZaliczonyLab? próbuję R1: potrzebuję OddałaZadania i ZdałaKolokwium
    OddałaZadania: fakt ✓
    ZdałaKolokwium: fakt ✓
  ZaliczonyLab: TAK (R1)
Dopuszczona: TAK (R2)
True

ZaliczonyPrzedmiot? próbuję R4: potrzebuję Dopuszczona i ZdałaEgzamin
  Dopuszczona? próbuję R2: potrzebuję ZaliczonyLab
    ZaliczonyLab? próbuję R1: potrzebuję OddałaZadania i ZdałaKolokwium
      OddałaZadania: fakt ✓
      ZdałaKolokwium: fakt ✓
    ZaliczonyLab: TAK (R1)
  Dopuszczona: TAK (R2)
  ZdałaEgzamin: brak faktu i reguły ✗
ZaliczonyPrzedmiot: NIE
False



**Odpowiedzi (2.1):**
1. …
2. Poniewaz w bazie wiedzy brakuje faktu.
3. print(wnioskuj_wstecz("MożeOddaćProjekt", FAKTY, REGULY))

### Zadanie 2.2. Nawrót: druga droga do celu

Bartek oddał zadania, ale nie zdał kolokwium. Na szczęście regulamin mówi: kto ma zwolnienie z kolokwium, też zalicza laboratorium. To reguła **R5** poniżej.

1. Uruchom komórkę. Bartek nie jest dopuszczony.
2. Dopisz Bartkowi fakt o zwolnieniu. Uruchom jeszcze raz i znajdź w wydruku miejsce, w którym algorytm **wraca** po porażce R1 i próbuje R5.

In [20]:
REGULY_Z_R5 = REGULY + [("R5", ["OddałaZadania", "MaZwolnienieZKolokwium"], "ZaliczonyLab")]

FAKTY_BARTKA = {
    "OddałaZadania","MaZwolnienieZKolokwium",
    # TODO: dopisz fakt o zwolnieniu (musi pasować do przesłanki R5)
}

wnioskuj_wstecz("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5)

Dopuszczona? próbuję R2: potrzebuję ZaliczonyLab
  ZaliczonyLab? próbuję R1: potrzebuję OddałaZadania i ZdałaKolokwium
    OddałaZadania: fakt ✓
    ZdałaKolokwium: brak faktu i reguły ✗
  ZaliczonyLab? próbuję R5: potrzebuję OddałaZadania i MaZwolnienieZKolokwium
    OddałaZadania: fakt ✓
    MaZwolnienieZKolokwium: fakt ✓
  ZaliczonyLab: TAK (R5)
Dopuszczona: TAK (R2)


True

In [21]:
# Test 2.2
assert wnioskuj_wstecz("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5, pokaz=False), "Bartek powinien być dopuszczony"
assert "ZdałaKolokwium" not in FAKTY_BARTKA, "Bartek nie zdał kolokwium: nie dopisuj tego faktu"
ok("Zadanie 2.2")

✅ Zadanie 2.2: wszystko działa


Moduł wyjaśnień pokazuje ten sam dowód krótko, jak w systemach ekspertowych. Uruchom:

In [22]:
for zdanie in wyjasnij("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5) or ["Nie da się udowodnić. Wróć do zadania 2.2."]:
    print(zdanie)

Dopuszczona, bo R2: ZaliczonyLab
ZaliczonyLab, bo R5: OddałaZadania i MaZwolnienieZKolokwium
OddałaZadania: fakt
MaZwolnienieZKolokwium: fakt


### Zadanie 2.3. Kto robi mniej pracy?

Funkcja `porownaj_algorytmy` buduje dużą bazę i liczy, ile razy każdy algorytm próbował zastosować regułę. Uruchom ją dla kilku rozmiarów bazy i uzupełnij tabelę w komórce tekstowej.

In [31]:
print(porownaj_algorytmy(lancuchy=10,  dlugosc=10))
print(porownaj_algorytmy(lancuchy=100, dlugosc=10))
print(porownaj_algorytmy(lancuchy=1000, dlugosc=10))
# TODO: uruchom jeszcze dla lancuchy=1000

{'reguł w bazie': 100, 'faktów wywnioskowanych w przód': 100, 'próby w przód': 1100, 'próby wstecz': 10}
{'reguł w bazie': 1000, 'faktów wywnioskowanych w przód': 1000, 'próby w przód': 11000, 'próby wstecz': 10}
{'reguł w bazie': 10000, 'faktów wywnioskowanych w przód': 10000, 'próby w przód': 110000, 'próby wstecz': 10}


| Łańcuchy | Reguł w bazie | Próby w przód | Próby wstecz |
|---|---|---|---|
| 10 | | | |
| 100 | | | |
| 1000 | | | |

**Pytania:**
1. Jak rośnie praca każdego algorytmu, gdy baza rośnie?

Liniowo.

2. Kiedy wnioskowanie w przód będzie jednak lepszym wyborem? (slajd 31)

Wnioskowanie w przod jest lepsze, gdy nie znamy konkretnego celu , ktory chcemy osiagnąc.

### ★ Zadanie 2.4. Pułapka: cykl

Ktoś dopisał dwie reguły, które wskazują na siebie nawzajem. Uruchom komórkę. Znajdź w kodzie `wnioskuj_wstecz` linie, które chronią przed nieskończoną rekurencją. Co by się stało bez nich? (Jeśli chcesz sprawdzić: skopiuj funkcję, usuń ochronę i uruchom. Zobaczysz `RecursionError`.)

In [35]:
R_cykl = [("C1", ["B"], "A"),
          ("C2", ["A"], "B")]
wnioskuj_wstecz("A", {"X"}, R_cykl)

A? próbuję C1: potrzebuję B
  B? próbuję C2: potrzebuję A
    A: cykl, przerywam ✗
  B: NIE
A: NIE


False

---
# Część 3. Wielu studentów. Świat zamknięty i otwarty

In [25]:
# ===== Gotowe narzędzia: pytania o wielu studentów =====

# Każdy student ma swój zbiór faktów. Reguły (REGULY) są wspólne dla wszystkich.
STUDENCI = {
    "ania":   {"OddałaZadania", "ZdałaKolokwium"},
    "bartek": {"OddałaZadania"},
    "celina": {"OddałaZadania", "ZdałaKolokwium", "ZdałaEgzamin"},
    "darek":  set(),          # o Darku nic nie wiemy
}

# Fakty, o których wiemy na pewno, że są FAŁSZEM (np. z protokołu)
WIADOMO_FALSZ = {
    "bartek": {"ZdałaKolokwium"},
}


def kto(cel, studenci=STUDENCI, reguly=REGULY):
    '''
    Odpowiednik pytania Prologu ze zmienną, np. ?- dopuszczona(Kto).  (slajd 34)
    Zwraca posortowaną listę studentów, dla których cel da się udowodnić.
    '''
    return sorted(s for s, fakty in studenci.items()
                  if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False))


def na_pewno_falsz(cel, falsz, reguly=REGULY):
    '''
    Sprawdza, czy cel jest NA PEWNO fałszem.
    Tak jest, gdy:
      - cel jest na liście faktów fałszywych, albo
      - są reguły z tym wnioskiem i KAŻDA z nich ma przesłankę, która jest na pewno fałszem.
    Upraszczamy: zakładamy, że nasze reguły to jedyne drogi do wniosku.
    '''
    if cel in falsz:
        return True
    kandydaci = [r for r in reguly if r[2] == cel]
    if not kandydaci:
        return False          # nie ma faktu ani reguły: nie wiemy
    return all(any(na_pewno_falsz(p, falsz, reguly) for p in przeslanki)
               for _, przeslanki, _ in kandydaci)


def zapytaj(student, cel, reguly=REGULY):
    '''
    Odpowiedź w świecie otwartym (slajd 32): "TAK", "NIE" albo "NIE WIADOMO".
    '''
    fakty = STUDENCI[student]
    falsz = WIADOMO_FALSZ.get(student, set())
    if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False):
        return "TAK"
    if na_pewno_falsz(cel, falsz, reguly):
        return "NIE"
    return "NIE WIADOMO"


def tabela(cele=("Dopuszczona", "ZaliczonyPrzedmiot")):
    '''Pokazuje obok siebie odpowiedzi w świecie zamkniętym i otwartym.'''
    wiersze = []
    for s in STUDENCI:
        for c in cele:
            zamkniety = "TAK" if wnioskuj_wstecz(c, STUDENCI[s], REGULY, pokaz=False) else "NIE"
            wiersze.append({"student": s, "pytanie": c,
                            "świat zamknięty": zamkniety, "świat otwarty": zapytaj(s, c)})
    return pd.DataFrame(wiersze)

print("Narzędzia z części 3 gotowe")

Narzędzia z części 3 gotowe


### Zadanie 3.1. `?- dopuszczona(Kto).`

1. Przewidź: kto jest dopuszczony, a kto zaliczył przedmiot? Potem uruchom.
2. Dopisz do `STUDENCI` nową studentkę `"ewa"`, która jest **dopuszczona**, ale **nie zaliczyła** przedmiotu.

In [28]:
STUDENCI["ewa"] = {"Dopuszczona", "NieZaliczonyPrzedmiot"}
# TODO: STUDENCI["ewa"] = {...}


print("?- dopuszczona(Kto).        ", kto("Dopuszczona"))
print("?- zaliczony_przedmiot(Kto).", kto("ZaliczonyPrzedmiot"))

?- dopuszczona(Kto).         ['ania', 'celina', 'ewa']
?- zaliczony_przedmiot(Kto). ['celina']


In [29]:
# Test 3.1
assert "ewa" in kto("Dopuszczona"), "Ewa powinna być dopuszczona"
assert "ewa" not in kto("ZaliczonyPrzedmiot"), "Ewa nie powinna mieć zaliczonego przedmiotu"
ok("Zadanie 3.1")

✅ Zadanie 3.1: wszystko działa


### Zadanie 3.2. „NIE” czy „NIE WIADOMO”?

Uruchom tabelę. Porównaj kolumny: gdzie świat zamknięty i otwarty dają różne odpowiedzi?

Potem dowiadujemy się z protokołu, że **Darek nie oddał zadań**. Dopisz to do `WIADOMO_FALSZ` i uruchom tabelę jeszcze raz. Jak zmieniły się odpowiedzi dla Darka?

In [37]:
# TODO: WIADOMO_FALSZ["darek"] = {...}
WIADOMO_FALSZ["darek"] = {"Dopuszczona", "ZaliczonyPrzedmiot" }

tabela()

,student,pytanie,świat zamknięty,świat otwarty
0,ania,Dopuszczona,TAK,TAK
1,ania,ZaliczonyPrzedmiot,NIE,NIE WIADOMO
2,bartek,Dopuszczona,NIE,NIE
3,bartek,ZaliczonyPrzedmiot,NIE,NIE
4,celina,Dopuszczona,TAK,TAK
5,celina,ZaliczonyPrzedmiot,TAK,TAK
6,darek,Dopuszczona,NIE,NIE
7,darek,ZaliczonyPrzedmiot,NIE,NIE
8,ewa,Dopuszczona,TAK,TAK
9,ewa,ZaliczonyPrzedmiot,NIE,NIE WIADOMO


In [38]:
# Test 3.2
assert zapytaj("darek", "Dopuszczona") == "NIE", "Po dopisaniu faktu z protokołu Darek powinien dostać NIE"
assert zapytaj("ania", "ZaliczonyPrzedmiot") == "NIE WIADOMO"
ok("Zadanie 3.2")

✅ Zadanie 3.2: wszystko działa


**Pytanie (3.2):** Podaj jeden system, w którym słusznie przyjmujemy świat zamknięty, i jeden, w którym byłoby to błędem.

Odpowiedź: …

---
# Część 4. Ontologia jako trójki i reasoner

Każdą wiedzę da się zapisać jako trójki **(podmiot, predykat, obiekt)**, jak w RDF (slajd 58). Poniżej mała ontologia uczelni. Część pierwsza to **schemat** (klasy i cechy relacji), część druga to **dane** (konkretne osoby i przedmioty).

In [39]:
ONTOLOGIA = {
    # hierarchia klas: "jest rodzajem" (is-a)
    ("Student",      "subClassOf", "Osoba"),
    ("Pracownik",    "subClassOf", "Osoba"),
    ("Prowadzący",   "subClassOf", "Pracownik"),
    ("Wykład",       "subClassOf", "Zajęcia"),
    ("Laboratorium", "subClassOf", "Zajęcia"),
    # relacja „prowadzi” łączy Prowadzącego (domain) z Przedmiotem (range)
    ("prowadzi", "domain", "Prowadzący"),
    ("prowadzi", "range",  "Przedmiot"),
    # cechy relacji (slajd 57)
    ("wymaga",        "type",      "TransitiveProperty"),   # przechodnia
    ("prowadzi",      "inverseOf", "prowadzonyPrzez"),      # odwrotna
    ("współpracujeZ", "type",      "SymmetricProperty"),    # symetryczna
}

DANE = {
    ("ania",   "type", "Student"),
    ("bartek", "type", "Student"),
    ("ania",   "zapisanaNa", "psi"),
    ("kowalski",   "prowadzi", "psi"),
    ("nowak",      "prowadzi", "mat_dyskretna"),
    ("wisniewska", "prowadzi", "logika"),
    ("psi",           "wymaga", "mat_dyskretna"),
    ("mat_dyskretna", "wymaga", "logika"),
    ("ania", "współpracujeZ", "bartek"),
}

T = ONTOLOGIA | DANE
print(len(T), "trójek")

19 trójek


In [41]:
# ===== Gotowe narzędzia: reasoner i zapytania =====

def jeden_krok(T):
    '''
    Jeden krok reasonera: stosuje 7 reguł do wszystkich par trójek
    i zwraca zbiór NOWYCH trójek (których jeszcze nie ma w T).

    Reguły (x, y, z to obiekty; A, B, C to klasy; p, q to relacje):
      1. (A subClassOf B), (B subClassOf C)          -> (A subClassOf C)
      2. (x type A), (A subClassOf B)                -> (x type B)
      3. (p domain C), (x p y)                       -> (x type C)
      4. (p range C),  (x p y)                       -> (y type C)
      5. (p type TransitiveProperty), (x p y), (y p z) -> (x p z)
      6. (p inverseOf q), (x p y)                    -> (y q x)
      7. (p type SymmetricProperty), (x p y)         -> (y p x)
    '''
    nowe = set()
    for (s, p, o) in T:                  # pierwsza trójka
        for (s2, p2, o2) in T:           # druga trójka
            if p == "subClassOf" and p2 == "subClassOf" and o == s2:      # reguła 1
                nowe.add((s, "subClassOf", o2))
            if p == "type" and p2 == "subClassOf" and o == s2:            # reguła 2
                nowe.add((s, "type", o2))
            if p == "domain" and p2 == s:                                 # reguła 3
                nowe.add((s2, "type", o))
            if p == "range" and p2 == s:                                  # reguła 4
                nowe.add((o2, "type", o))
            if p == "inverseOf" and p2 == s:                              # reguła 6
                nowe.add((o2, o, s2))
            if p == "type" and o == "SymmetricProperty" and p2 == s:      # reguła 7
                nowe.add((o2, s, s2))
        # reguła 5: relacja przechodnia, potrzebujemy trzeciej trójki (y p z)
        if (p, "type", "TransitiveProperty") in T:
            for (s3, p3, o3) in T:
                if p3 == p and s3 == o:
                    nowe.add((s, p, o3))
    return nowe - T


def reasoner(T, pokaz=True):
    '''
    Reasoner: powtarza jeden_krok, aż nic nowego się nie pojawi (punkt stały).
    To jest wnioskowanie w przód z części 1, tylko na trójkach.
    Zwraca zbiór wszystkich trójek: podanych i wywnioskowanych.
    '''
    T = set(T)
    krok = 0
    while True:
        krok += 1
        nowe = jeden_krok(T)
        if pokaz:
            print(f"Krok {krok}: {len(nowe)} nowych trójek")
        if not nowe:
            return T
        T |= nowe


def znajdz_sprzecznosci(T):
    '''
    Szuka obiektów, które należą do dwóch klas rozłącznych (slajd 60).
    Dla każdej trójki (A, disjointWith, B) sprawdza, czy jakiś x ma typ A i typ B.
    Zwraca listę (x, A, B). Pusta lista = brak sprzeczności.
    '''
    wynik = []
    for (A, p, B) in T:
        if p == "disjointWith":
            for (x, p2, klasa) in T:
                if p2 == "type" and klasa == A and (x, "type", B) in T:
                    wynik.append((x, A, B))
    return sorted(wynik)


def szukaj(T, s=None, p=None, o=None):
    '''
    Wzorzec trójki z dziurami, jak w SPARQL (slajd 67).
    None oznacza "dowolna wartość" (zmienna).
    Przykład: szukaj(T, p="prowadzi") -> wszystkie trójki z relacją "prowadzi".
    Zwraca posortowaną listę pasujących trójek.
    '''
    return sorted(t for t in T
                  if (s is None or t[0] == s)
                  and (p is None or t[1] == p)
                  and (o is None or t[2] == o))


def pokaz_nowe(T_przed, T_po):
    '''Wypisuje w tabeli tylko te trójki, które dopisał reasoner.'''
    return pd.DataFrame(sorted(T_po - T_przed), columns=["podmiot", "predykat", "obiekt"])

print("Narzędzia z części 4 gotowe")

Narzędzia z części 4 gotowe


### Zadanie 4.1. Co wywnioskuje reasoner?

1. Uruchom reasoner i przejrzyj nowe trójki.
2. Przy trzech trójkach poniżej wpisz numer reguły z komentarza w `jeden_krok`, która ją dała. Uwaga: jedna z nich wymaga dwóch reguł po kolei.
   - `(psi, wymaga, logika)` …
   - `(psi, prowadzonyPrzez, kowalski)` …
   - `(kowalski, type, Osoba)` …

In [42]:
T_pelne = reasoner(T)
pokaz_nowe(T, T_pelne)

Krok 1: 14 nowych trójek
Krok 2: 6 nowych trójek
Krok 3: 0 nowych trójek


,podmiot,predykat,obiekt
0,Prowadzący,subClassOf,Osoba
1,ania,type,Osoba
2,bartek,type,Osoba
3,bartek,współpracujeZ,ania
4,kowalski,type,Osoba
5,kowalski,type,Pracownik
6,kowalski,type,Prowadzący
7,logika,prowadzonyPrzez,wisniewska
8,logika,type,Przedmiot
9,mat_dyskretna,prowadzonyPrzez,nowak


Odpowiedzi: …

### Zadanie 4.2. Rozbuduj ontologię

Dopisz do `NOWE` trójki opisujące:
- klasę **Doktorant**, która jest podklasą Studenta,
- doktoranta **zielinski**, który prowadzi przedmiot **lab_psi**,
- przedmiot **lab_psi**, który wymaga **psi**.

Nie dopisuj, że zielinski jest Osobą ani że jest Prowadzącym. Niech reasoner sam to znajdzie.

In [45]:
NOWE = set([
   ("Doktorant", "subClassOf", "Student"),
    ("zielinski", "type", "Doktorant"),
    ("zielinski", "prowadzi", "lab_psi"),
    ("lab_psi", "wymaga", "psi"),
])

T2 = reasoner(T | NOWE)
print(szukaj(T2, s="zielinski", p="type"))
print(szukaj(T2, s="lab_psi", p="wymaga"))

Krok 1: 20 nowych trójek
Krok 2: 9 nowych trójek
Krok 3: 0 nowych trójek
[('zielinski', 'type', 'Doktorant'), ('zielinski', 'type', 'Osoba'), ('zielinski', 'type', 'Pracownik'), ('zielinski', 'type', 'Prowadzący'), ('zielinski', 'type', 'Student')]
[('lab_psi', 'wymaga', 'logika'), ('lab_psi', 'wymaga', 'mat_dyskretna'), ('lab_psi', 'wymaga', 'psi')]


In [46]:
# Test 5.2
assert ("zielinski", "type", "Osoba") not in NOWE, "Tego ma się domyślić reasoner"
for t in [("zielinski", "type", "Osoba"), ("zielinski", "type", "Prowadzący"),
          ("zielinski", "type", "Student"), ("lab_psi", "wymaga", "logika")]:
    assert t in T2, f"Reasoner nie wywnioskował {t}. Sprawdź swoje trójki."
ok("Zadanie 4.2")

✅ Zadanie 4.2: wszystko działa


### Zadanie 4.3. Reasoner łapie sprzeczności

1. Ktoś przy imporcie danych zapisał PSI jednocześnie jako Wykład i Salę (slajd 60). Uruchom pierwszą komórkę. Dlaczego sprzeczność widać dopiero **po** reasonerze?
2. Dopisz w drugiej komórce, że **Student** i **Prowadzący** to klasy rozłączne. Uruchom. Kto jest teraz sprzeczny? Czy to błąd w danych, czy w ontologii?

In [47]:
T_bledne = T | {
    ("Zajęcia", "disjointWith", "Sala"),   # schemat: zajęcia to nie sala
    ("psi", "type", "Wykład"),             # błąd przy imporcie
    ("psi", "type", "Sala"),               # błąd przy imporcie
}
print("Przed reasonerem:", znajdz_sprzecznosci(T_bledne))
print("Po reasonerze:   ", znajdz_sprzecznosci(reasoner(T_bledne, pokaz=False)))

Przed reasonerem: []
Po reasonerze:    [('psi', 'Zajęcia', 'Sala')]


In [52]:
ROZLACZNE = set([
    ("Zajęcia", "disjointWith", "Sala"),
    ("psi", "disjointWith", "Wykład"),
    ("psi", "disjointWith", "Sala"),
])
print(znajdz_sprzecznosci(reasoner(T | NOWE | ROZLACZNE, pokaz=False)))

[]


Odpowiedzi: …

### Zadanie 4.4. Pytania do grafu: wzorce trójek

Funkcja `szukaj` działa jak proste zapytanie SPARQL: `None` to dziura. Uzupełnij wywołania tak, żeby odpowiedzieć na pytania. Korzystaj z `T_pelne`.

1. Jakie typy ma kowalski? (gotowe, przykład)
2. Kto prowadzi jakieś zajęcia? (wszystkie trójki z relacją `prowadzi`)
3. Jakie przedmioty trzeba zaliczyć przed PSI?
4. **Pytanie kompetencyjne** ze slajdu 62: kto prowadzi przedmioty, które ania musi zaliczyć przed PSI? Dwa kroki są gotowe, uzupełnij tylko wzorce.

In [56]:
print("1.", szukaj(T_pelne, s="kowalski", p="type"))
print("2.", szukaj(T_pelne, p="prowadzi"))
print("3.", szukaj(T_pelne, o="psi"))

# 4. Krok 1: przedmioty wymagane przed PSI (bierzemy obiekt trójki: t[2])
przedmioty = [t[2] for t in szukaj(T_pelne, s="psi", p="wymaga")]
# Krok 2: dla każdego przedmiotu: kto go prowadzi (bierzemy podmiot trójki: t[0])
prowadzacy = sorted(t[0] for z in przedmioty for t in szukaj(T_pelne, p="prowadzi", o=z))
print("4.", prowadzacy)

1. [('kowalski', 'type', 'Osoba'), ('kowalski', 'type', 'Pracownik'), ('kowalski', 'type', 'Prowadzący')]
2. [('kowalski', 'prowadzi', 'psi'), ('nowak', 'prowadzi', 'mat_dyskretna'), ('wisniewska', 'prowadzi', 'logika')]
3. [('ania', 'zapisanaNa', 'psi'), ('kowalski', 'prowadzi', 'psi')]
4. ['nowak', 'wisniewska']


In [57]:
# Test 5.4
assert sorted(przedmioty) == ["logika", "mat_dyskretna"], "Krok 1: sprawdź wzorzec (psi, wymaga, ?)"
assert prowadzacy == ["nowak", "wisniewska"], "Krok 2: sprawdź wzorzec (?, prowadzi, z)"
ok("Zadanie 4.4")

✅ Zadanie 4.4: wszystko działa


**Pytanie:** Czy wynik pytania 4 byłby taki sam na `T` (przed reasonerem)? Dlaczego?

Odpowiedź: …

### Zadanie 4.5. Wyjątki: najbliższa informacja wygrywa

Logika klasyczna nie radzi sobie z pingwinem (slajd 52). Sieć semantyczna szuka cechy w węźle, a jeśli jej tam nie ma, idzie strzałką „jest rodzajem” w górę (slajd 54). Przeczytaj funkcję `cecha`.

1. Uruchom. Sprawdź wynik dla Tweety i Pingu.
2. Dopisz do grafu **strusia Bolka** (Struś to rodzaj Ptaka, strusie nie latają, ale szybko biegają).
3. Dopisz wyjątek na poziomie pojedynczego obiektu: wróbel **Rysio** ma złamane skrzydło i nie lata.

In [58]:
JEST_RODZAJEM = {          # dziecko -> rodzic (strzałka "jest rodzajem" albo "jest")
    "Tweety":  "Wróbel",
    "Pingu":   "Pingwin",
    "Wróbel":  "Ptak",
    "Pingwin": "Ptak",
    "Ptak":    "Zwierzę",
}
CECHY = {                  # węzeł -> jego własne cechy
    "Zwierzę": {"oddycha": True},
    "Ptak":    {"lata": True, "ma_pióra": True},
    "Pingwin": {"lata": False, "pływa": True},
}

def cecha(wezel, nazwa, pokaz=False):
    '''
    Szuka cechy `nazwa` dla węzła `wezel` w sieci semantycznej.
    Zaczyna od samego węzła i idzie w górę po JEST_RODZAJEM.
    Zwraca pierwszą znalezioną wartość (najbliższa wygrywa) albo None.
    '''
    droga = []
    while wezel is not None:
        droga.append(wezel)
        if nazwa in CECHY.get(wezel, {}):          # cecha zapisana w tym węźle?
            if pokaz:
                print(" → ".join(droga), f": {nazwa} = {CECHY[wezel][nazwa]}")
            return CECHY[wezel][nazwa]
        wezel = JEST_RODZAJEM.get(wezel)           # nie ma: idziemy do rodzica
    if pokaz:
        print(" → ".join(droga), f": brak cechy {nazwa}")
    return None

cecha("Tweety", "lata", pokaz=True)
cecha("Pingu", "lata", pokaz=True)
cecha("Pingu", "oddycha", pokaz=True);

Tweety → Wróbel → Ptak : lata = True
Pingu → Pingwin : lata = False
Pingu → Pingwin → Ptak → Zwierzę : oddycha = True


In [67]:
JEST_RODZAJEM["Bolek"] = "Ptak"
CECHY["Bolek"] = {"lata": False, "ma_pióra": True}

JEST_RODZAJEM["Rysio"] = "Wróbel"
CECHY["Rysio"] = {"lata": False}

for kto_ in ["Tweety", "Pingu", "Bolek", "Rysio"]:
    cecha(kto_, "lata", pokaz=True)

Tweety → Wróbel → Ptak : lata = True
Pingu → Pingwin : lata = False
Bolek : lata = False
Rysio : lata = False


In [68]:
# Test 5.5
assert cecha("Bolek", "lata") is False and cecha("Bolek", "ma_pióra") is True, "Bolek: struś nie lata, ale jako ptak ma pióra"
assert cecha("Rysio", "lata") is False, "Rysio: wyjątek zapisz w CECHY dla samego Rysia"
assert cecha("Tweety", "lata") is True
ok("Zadanie 4.5")

✅ Zadanie 4.5: wszystko działa


### ★ Zadanie 4.6. Prawdziwy SPARQL: biblioteka `rdflib`

Te same trójki wczytujemy do biblioteki `rdflib` i pytamy prawdziwym SPARQL-em. Zapytanie odpowiada na pytanie kompetencyjne z zadania 4.4. Porównaj je z dwoma krokami w Pythonie: wspólna zmienna `?przedmiot` łączy dwa wzorce.

Zmień zapytanie tak, żeby pokazywało, **kogo uczy kowalski** (reguła z wykładu: Y prowadzi P, X jest zapisany na P → Y uczy X).

In [ ]:
!pip -q install rdflib

In [ ]:
from rdflib import Graph, Namespace

EX = Namespace("http://example.org/uczelnia/")   # prefiks: każda nazwa staje się adresem URI
g = Graph()
for s, p, o in T_pelne:
    g.add((EX[s], EX[p], EX[o]))                  # nasze trójki -> trójki RDF
print(len(g), "trójek w grafie")

zapytanie = '''
PREFIX ex: <http://example.org/uczelnia/>
SELECT ?kto ?przedmiot WHERE {
    ex:psi ex:wymaga ?przedmiot .      # wzorzec 1
    ?kto ex:prowadzi ?przedmiot .      # wzorzec 2 (ta sama zmienna ?przedmiot)
}
'''
for wiersz in g.query(zapytanie):
    print([str(x).split("/")[-1] for x in wiersz])

# TODO ★: zapytanie "kogo uczy kowalski"

---
# Część 5. Wikidata i SPARQL

Teraz prawdziwy graf wiedzy: Wikidata, ponad 100 milionów elementów. `Q…` to element (rzecz), `P…` to właściwość (relacja) (slajd 66).

| Identyfikator | Znaczenie |
|---|---|
| `wdt:P31` | jest to (instancja klasy) |
| `wdt:P279` | podklasa |
| `wdt:P17` | państwo |
| `wdt:P131` | leży w jednostce administracyjnej |
| `wdt:P1082` | populacja |
| `wdt:P625` | współrzędne |
| `wdt:P27` | obywatelstwo |
| `wdt:P166` | otrzymana nagroda |
| `wd:Q515` | miasto |
| `wd:Q36` | Polska |
| `wd:Q183` | Niemcy |
| `wd:Q580` | Łódź |
| `wd:Q3918` | uniwersytet |
| `wd:Q7191` | Nagroda Nobla |

Identyfikatory innych rzeczy znajdziesz na [wikidata.org](https://www.wikidata.org) (wyszukiwarka u góry strony).

In [69]:
# ===== Gotowe narzędzia: Wikidata =====
import requests, urllib.parse

def wikidata(zapytanie):
    '''
    Wysyła zapytanie SPARQL do Wikidata i zwraca wynik jako tabelę pandas.
    Gdy połączenie się nie uda, wypisuje błąd i zwraca pustą tabelę.
    '''
    try:
        r = requests.get("https://query.wikidata.org/sparql",
                         params={"query": zapytanie, "format": "json"},
                         headers={"User-Agent": "PSI-lab3-student/1.0 (zajecia dydaktyczne)"},
                         timeout=60)
        r.raise_for_status()                    # błąd HTTP -> wyjątek
    except Exception as e:
        print("Nie udało się połączyć z Wikidata:", e)
        return pd.DataFrame()
    dane = r.json()
    kolumny = dane["head"]["vars"]              # nazwy zmiennych z SELECT
    # Każdy wiersz wyniku to słownik {zmienna: wartość}; brakujące wartości -> None
    wiersze = [{k: (w[k]["value"] if k in w else None) for k in kolumny}
               for w in dane["results"]["bindings"]]
    return pd.DataFrame(wiersze, columns=kolumny)


def link(zapytanie):
    '''Zwraca link, który otwiera zapytanie w przeglądarce query.wikidata.org (np. żeby zobaczyć mapę).'''
    return "https://query.wikidata.org/#" + urllib.parse.quote(zapytanie)

print("Narzędzia z części 5 gotowe")

Narzędzia z części 5 gotowe


### Zadanie 5.1. Zapytanie z wykładu

Uruchom zapytanie ze slajdu 67: polskie miasta powyżej 200 000 mieszkańców. Odpowiedz:
1. Czy jakieś miasto pojawia się kilka razy? Dlaczego?
2. Co robi `wdt:P31/wdt:P279*`? Z czym z części 4 to się kojarzy?

In [70]:
Q_MIASTA = '''
SELECT ?miasto ?miastoLabel ?ludnosc WHERE {
  ?miasto wdt:P31/wdt:P279* wd:Q515 ;    # jest miastem (albo podklasą miasta)
          wdt:P17 wd:Q36 ;               # leży w Polsce
          wdt:P1082 ?ludnosc .           # ma populację
  FILTER(?ludnosc > 200000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY DESC(?ludnosc)
'''
wikidata(Q_MIASTA)

,miasto,miastoLabel,ludnosc
0,http://www.wikidata.org/entity/Q3114019,Konurbacja górnośląska,1903000
1,http://www.wikidata.org/entity/Q270,Warszawa,1862402
2,http://www.wikidata.org/entity/Q270,Warszawa,1862402
3,http://www.wikidata.org/entity/Q270,Warszawa,1862402
4,http://www.wikidata.org/entity/Q270,Warszawa,1862402
5,http://www.wikidata.org/entity/Q270,Warszawa,1862402
6,http://www.wikidata.org/entity/Q270,Warszawa,1862402
7,http://www.wikidata.org/entity/Q31487,Kraków,804237
8,http://www.wikidata.org/entity/Q31487,Kraków,804237
9,http://www.wikidata.org/entity/Q1799,Wrocław,673531


Odpowiedzi:
1. Poniewaz w zapytaniu nie ma funkcji usuwajacej duplikaty (DISTINCT),  a system znajduje miasto na rozne sposoby

2.(1)Szuka obiektow bedacych danym typem oraz wszystkich jego podtypow

2.(2)Kojarzy sie z dziedziczeniem klas z czesci 4

### Zadanie 5.2. Zmień zapytanie

Skopiowane zapytanie jest poniżej. Zmień **dwie rzeczy**: państwo na Niemcy i próg na 500 000. Potem otwórz wersję z mapą przez link.

In [71]:
Q_NIEMCY = '''
SELECT DISTINCT ?miasto ?miastoLabel ?ludnosc WHERE {
  ?miasto wdt:P31/wdt:P279* wd:Q515 ;
          wdt:P17 wd:Q36 ;               # TODO: Niemcy
          wdt:P1082 ?ludnosc .
  FILTER(?ludnosc > 200000)              # TODO: 500 000
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY DESC(?ludnosc)
'''
display(wikidata(Q_NIEMCY))

# Wersja z mapą: linia #defaultView:Map i współrzędne ?wsp. Kliknij link.
Q_MAPA = Q_NIEMCY.replace("SELECT DISTINCT ?miasto ?miastoLabel ?ludnosc",
                          "#defaultView:Map\nSELECT DISTINCT ?miasto ?miastoLabel ?wsp") \
                 .replace("wdt:P1082 ?ludnosc .", "wdt:P1082 ?ludnosc ;\n          wdt:P625 ?wsp .")
print(link(Q_MAPA))

,miasto,miastoLabel,ludnosc
0,http://www.wikidata.org/entity/Q3114019,Konurbacja górnośląska,1903000
1,http://www.wikidata.org/entity/Q270,Warszawa,1862402
2,http://www.wikidata.org/entity/Q31487,Kraków,804237
3,http://www.wikidata.org/entity/Q1799,Wrocław,673531
4,http://www.wikidata.org/entity/Q580,Łódź,648711
5,http://www.wikidata.org/entity/Q268,Poznań,546859
6,http://www.wikidata.org/entity/Q1792,Gdańsk,487834
7,http://www.wikidata.org/entity/Q393,Szczecin,387700
8,http://www.wikidata.org/entity/Q37333,Lublin,334681
9,http://www.wikidata.org/entity/Q41252,Bydgoszcz,324984


https://query.wikidata.org/#%0A%23defaultView%3AMap%0ASELECT%20DISTINCT%20%3Fmiasto%20%3FmiastoLabel%20%3Fwsp%20WHERE%20%7B%0A%20%20%3Fmiasto%20wdt%3AP31/wdt%3AP279%2A%20wd%3AQ515%20%3B%0A%20%20%20%20%20%20%20%20%20%20wdt%3AP17%20wd%3AQ36%20%3B%20%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20TODO%3A%20Niemcy%0A%20%20%20%20%20%20%20%20%20%20wdt%3AP1082%20%3Fludnosc%20%3B%0A%20%20%20%20%20%20%20%20%20%20wdt%3AP625%20%3Fwsp%20.%0A%20%20FILTER%28%3Fludnosc%20%3E%20200000%29%20%20%20%20%20%20%20%20%20%20%20%20%20%20%23%20TODO%3A%20500%20000%0A%20%20SERVICE%20wikibase%3Alabel%20%7B%20bd%3AserviceParam%20wikibase%3Alanguage%20%22pl%2Cen%22.%20%7D%0A%7D%0AORDER%20BY%20DESC%28%3Fludnosc%29%0A


### Zadanie 5.3. Uczelnie w Łodzi i polscy nobliści

W obu zapytaniach brakuje identyfikatorów. Uzupełnij `Q____` i `P____` korzystając z tabeli na początku części 5.

Potem odpowiedz: czy na liście noblistów jest Maria Skłodowska-Curie? Jeśli nie, to dlaczego? Co to mówi o świecie otwartym?

In [72]:
Q_UCZELNIE = '''
SELECT DISTINCT ?uczelnia ?uczelniaLabel WHERE {
  ?uczelnia wdt:P31/wdt:P279* wd:Q____ ;    # TODO: jest uniwersytetem
            wdt:P131 wd:Q____ .             # TODO: leży w Łodzi
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY ?uczelniaLabel
'''
display(wikidata(Q_UCZELNIE))

Q_NOBLISCI = '''
SELECT DISTINCT ?osoba ?osobaLabel ?nagrodaLabel WHERE {
  ?osoba wdt:P____ wd:Q36 ;                 # TODO: obywatelstwo polskie
         wdt:P166 ?nagroda .                # dostał nagrodę
  ?nagroda wdt:P31|wdt:P279 wd:Q____ .      # TODO: ta nagroda to (jakaś) Nagroda Nobla
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY ?osobaLabel
'''
display(wikidata(Q_NOBLISCI))

,uczelnia,uczelniaLabel


,osoba,osobaLabel,nagrodaLabel


Odpowiedź: …

---
# Podsumowanie

Odpowiedz krótko (2–3 zdania na pytanie):

1. Który algorytm wybrałbyś do systemu, który pilnuje terminów w USOS i wysyła powiadomienia, a który do asystenta, który odpowiada na pytanie „czy mogę podejść do egzaminu”? Dlaczego?
2. W zadaniu 4.2 nikt nie wpisał, że zielinski jest Osobą. Skąd reasoner to wie? Gdzie jest „inteligencja”: w danych czy w algorytmie?
3. Wszystkie reguły, fakty i trójki z dzisiejszych zajęć ktoś wpisał ręcznie. Co by się stało, gdyby baza miała 100 000 reguł zebranych od wielu ekspertów? Jak można by zdobywać wiedzę inaczej? (To temat wykładu 5.)

Odpowiedzi:
1. Do USOS-a wybieramy wnioskowanie w przod, bo automatycznie przetwarza i wyciąga wszystkie wnioski o terminach. Do asystenta z konkretnym pytaniem lepsze jest wnioskowanie wstecz, poniewaz dziala szybciej i szuka odpowiedzi tylko dla konkretnego celu.

2. Reasoner wie ze zielinski to Osoba, za pomoca dziedziczenia klas (Doktorant -> Student -> Osoba).

3. Baza z 100 000 recznych regul stalaby sie niespojna i trudna w utrzymaniu przez kolizje miedzy ekspertami. Wiedze mozna zdobywac automatycznie np. poprzez ekstrakcje informacji z tekstów, uczenie maszynowe lub analize duzych baz danych.
---
*Gotowe? Zapisz notatnik (Plik → Pobierz → .ipynb) i oddaj go zgodnie z zasadami prowadzącego.*